# 12 — Avaliação no teste do SVM binário (com recall por tipo de ataque)

Avalia, **uma única vez**, o **modelo final (refit)** do SVM escolhido pela busca do notebook 11 (e, para a comparação, o **padrão**: C = 1, gamma = 1× o de referência, quando a escolhida for outra). O teste é lido da view `prep_vw_rf_bin_teste_sem_sobrep` (as **mesmas linhas** que o RF avaliou, sem as que se sobrepõem ao treino), **sem balancear e sem pesos**. É o equivalente do notebook 08 do RF, com as diferenças do SVM.

| O que | Como |
|---|---|
| **Escore e limiar** | O escore é a **distância à fronteira** (`decision_function`, não uma probabilidade). O limiar natural é **0**; os outros dois (falso alarme de 1% e de 0,1%) são **os gravados na validação** pelo notebook 11. **Nenhum limiar é escolhido no teste** |
| **Recall por tipo de ataque** | O tipo de cada linha vem da tabela `prep_final_teste` (junção por `id_linha`, lote a lote). Mostra também se o tipo **existe no treino** (por exemplo, o `Portmap` não existe) |
| **Comparação** | O padrão do SVM é avaliado nas mesmas linhas, no mesmo passo. A comparação é **descritiva**: a escolha já foi feita na validação cruzada |
| **Previsão** | O SVM é lento para prever (custa ~200 ns por par linha × vetor de suporte): cada lote é dividido em `PROCESSOS_PREVISAO` processos (2 deu ganho de 1,84× na calibração) |
| **Custo** | Vazão em dados reais, latência de 1 fluxo e vazão em lote, com e sem o pré-processamento |
| **Trava de uso único** | Repetir o teste depois de ajustar o modelo o invalida como avaliação independente. Se `metricas_teste.json` já existir, o notebook para |

## Tempo estimado
A previsão dos 7,2 milhões de fluxos leva **~20 a 35 min por modelo com 2 processos** (37 a 64 min com 1); com a comparação com o padrão são **~40 a 70 min**. A leitura do servidor soma uns 4 minutos. (Estimativas da calibração; o notebook mostra o tempo real na barra.) Para encurtar, `COMPARAR_COM_PADRAO = False`.

## Antes de rodar
O notebook 11 precisa ter terminado. Feche os outros notebooks (cada kernel guarda memória); com 2 processos de previsão, sobra menos RAM. O teste é lido em lotes de `LOTE_LINHAS` com checkpoint: se for interrompido, **rode a célula de novo**. **Não mude `LOTE_LINHAS` no meio de uma execução.**

In [1]:
import os, re, json, time, http.client, urllib.parse
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 120)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env_path = REPO_ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(f"Arquivo .env não encontrado em: {env_path}")
env = {**load_env(env_path), **os.environ}

CH_HOST, CH_PORT = env["CLICKHOUSE_HOST"], env["CLICKHOUSE_PORT"]
CH_USER, CH_PASSWORD, CH_DATABASE = env["CLICKHOUSE_USER"], env["CLICKHOUSE_PASSWORD"], env["CLICKHOUSE_DATABASE"]
CH_SCHEME = env.get("CLICKHOUSE_SCHEME", "https")
HTTP_TIMEOUT = 1800
print(f"ClickHouse: {CH_HOST}:{CH_PORT}/{CH_DATABASE}  |  usuário: {CH_USER}")

ClickHouse: clickhouse-production-e204.up.railway.app:443/railway  |  usuário: clickhouse


In [2]:
# ---------------- transporte HTTP direto (conexão reutilizada + reconexão automática só para leituras) ----------------
class ErroConexao(Exception):
    pass


CONEXAO_ESPERAS = (0, 5, 15, 45, 90)
_conn = None
ULT_STATS = None                  # estatísticas (linhas lidas etc.) da última consulta JSON


def _fechar_conexao():
    global _conn
    try:
        if _conn is not None:
            _conn.close()
    except Exception:
        pass
    _conn = None


def _http(query: str) -> pd.DataFrame:
    """Executa via HTTP. Repetir leituras é seguro; escritas não são reenviadas aqui (quem repete é carregar(), com verificação de contagem)."""
    global _conn, ULT_STATS
    params = {"database": CH_DATABASE, "default_format": "JSONCompact", "wait_end_of_query": 1}
    primeiro = re.match(r"\s*(\w+)", query)
    verbo = primeiro.group(1).upper() if primeiro else ""
    if verbo == "ALTER":
        params["mutations_sync"] = 2          # espera o DELETE terminar antes de responder
    caminho = "/?" + urllib.parse.urlencode(params)
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    somente_leitura = verbo not in {"INSERT", "CREATE", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE", "DELETE", "UPDATE"}
    tentativas = CONEXAO_ESPERAS if somente_leitura else (0,)
    ultimo = None
    for espera in tentativas:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s...")
            time.sleep(espera)
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            corpo = resp.read()
        except (OSError, http.client.HTTPException) as e:
            ultimo = e
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"Falha de conexão durante escrita: {e!r}") from e
            continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} [código ClickHouse: {resp.getheader('X-ClickHouse-Exception-Code')}] corpo={corpo[:300].decode('utf-8', 'replace')!r}")
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"HTTP {resp.status} durante escrita. Resposta: {corpo.decode('utf-8', 'replace')[:500]!r}")
            continue
        if resp.status != 200:
            raise Exception(corpo.decode("utf-8", "replace")[:2000])
        if not corpo.strip():
            return pd.DataFrame()
        dados = json.loads(corpo)
        ULT_STATS = dados.get("statistics")
        return pd.DataFrame(dados.get("data", []), columns=[m["name"] for m in dados.get("meta", [])])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(tentativas)} tentativas: {ultimo!r}")

In [3]:
# ---------------- TRAVA: só é possível escrever em tabelas prep_* ----------------
PREFIXO_NOVO = "prep_"          # tabelas que ESTE notebook pode criar / alimentar / apagar

_LITERAL = re.compile(r"'[^'\\]*(?:\\.[^'\\]*)*'")      # literais entre aspas simples (aceita \' escapado)


def _sem_literais(s: str) -> str:
    # texto SQL sem o conteúdo dos literais: ';' ou 'DROP TABLE' dentro de aspas NÃO contam na checagem da trava
    return _LITERAL.sub("''", s)


_VERBOS_ESCRITA = ("CREATE", "INSERT", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE")
_VERBOS_PROIBIDOS = ("DELETE", "UPDATE", "ATTACH", "DETACH", "EXCHANGE", "GRANT", "REVOKE", "KILL", "SYSTEM", "SET", "USE")
_ALVO = re.compile(r"\b(?:TABLE|INTO|VIEW)\s+(?:IF\s+(?:NOT\s+)?EXISTS\s+)?(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)
_RENAME = re.compile(r"(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)\s+TO\s+(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)


def _tabela_de(ident: str) -> str:
    # 'banco.tabela' ou '`tabela`' -> 'tabela' (e exige que o banco, se informado, seja o atual)
    partes = [p.strip("`") for p in re.findall(r"`[^`]+`|[^.]+", ident)]
    if len(partes) == 2 and partes[0] != CH_DATABASE:
        raise PermissionError(f"Escrita bloqueada: banco '{partes[0]}' diferente do atual '{CH_DATABASE}'.")
    return partes[-1]


def validar_escrita(sql: str, permitir_prefixos=()) -> None:
    # Levanta PermissionError se o comando escreveria fora das tabelas permitidas. Leituras passam sem restrição.
    s = sql.strip().rstrip(";").strip()
    s_chk = _sem_literais(s)
    if ";" in s_chk:
        raise PermissionError("Escrita bloqueada: envie uma instrução por vez.")
    verbo = (re.match(r"\s*(\w+)", s).group(1) if re.match(r"\s*(\w+)", s) else "").upper()
    if verbo in _VERBOS_PROIBIDOS:
        raise PermissionError(f"Comando '{verbo}' não é permitido neste notebook.")
    if verbo not in _VERBOS_ESCRITA:
        return                                                    # SELECT / SHOW / DESCRIBE / EXISTS ...
    prefixos = (PREFIXO_NOVO, *permitir_prefixos)
    if verbo == "RENAME":
        nomes = [_tabela_de(x) for par in _RENAME.findall(s_chk) for x in par]
    else:
        nomes = [_tabela_de(x) for x in _ALVO.findall(s_chk)]
    if not nomes:
        raise PermissionError(f"Escrita bloqueada: não identifiquei a tabela alvo em: {s[:80]}...")
    ruins = [n for n in nomes if not n.startswith(prefixos)]
    if ruins:
        raise PermissionError(f"Escrita bloqueada em {ruins}: só é permitido escrever em tabelas com prefixo {prefixos}.")


def executar(sql: str, permitir_prefixos=()) -> pd.DataFrame:
    validar_escrita(sql, permitir_prefixos)
    return _http(sql)


consultar = executar      # leituras também passam pela trava (e ela as libera)

# autoteste da trava (não toca no servidor)
_deve_passar = ["SELECT 1", "SHOW CREATE TABLE data2_portmap", f"CREATE TABLE {PREFIXO_NOVO}x AS data2_portmap",
                f"INSERT INTO {PREFIXO_NOVO}x SELECT * FROM data2_portmap LIMIT 1", f"DROP TABLE IF EXISTS `{PREFIXO_NOVO}x`",
                f"INSERT INTO {PREFIXO_NOVO}x VALUES ('idêntica a a; teste', 'DROP TABLE data_syn')"]
_deve_bloquear = ["INSERT INTO data2_portmap SELECT 1", "DROP TABLE data2_portmap", "DROP TABLE IF EXISTS data_tftp",
                  "TRUNCATE TABLE data_syn", "ALTER TABLE data_syn DROP COLUMN label", "CREATE TABLE outra AS data_syn",
                  "RENAME TABLE data_syn TO prep_syn", "CREATE DATABASE x", "DELETE FROM data_syn WHERE 1",
                  f"INSERT INTO {PREFIXO_NOVO}x SELECT 1; DROP TABLE data_syn", "SYSTEM DROP MARK CACHE", f"INSERT INTO {PREFIXO_NOVO}x VALUES ('a'); DROP TABLE data_syn"]
for _s in _deve_passar:
    validar_escrita(_s)
for _s in _deve_bloquear:
    try:
        validar_escrita(_s); raise AssertionError(f"a trava deixou passar: {_s}")
    except PermissionError:
        pass
print(f"Trava de segurança OK: {len(_deve_passar)} comandos permitidos e {len(_deve_bloquear)} bloqueados no autoteste.")

Trava de segurança OK: 6 comandos permitidos e 12 bloqueados no autoteste.


In [4]:
# ---------- ferramentas gerais ----------
import os, sys, json, math, time, zlib, platform, threading
from pathlib import Path
from datetime import datetime
import numpy as np
try:
    import psutil
except ImportError:
    psutil = None
    print("⚠ psutil não está instalado: o pico de memória e o hardware não serão registrados. Rode `pip install psutil` e execute esta célula de novo.")
try:
    import sklearn, joblib
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
except ImportError as e:
    raise ImportError("Faltam bibliotecas: rode `pip install scikit-learn joblib psutil` e execute esta célula de novo.") from e

def lit(s): return "'" + str(s).replace("\\", "\\\\").replace("'", "\\'") + "'"
def q(c): return "`" + str(c).replace("`", "``") + "`"

def consultar_num(sql):
    df = consultar(sql)
    for c in df.columns:
        try: df[c] = pd.to_numeric(df[c])
        except (ValueError, TypeError): pass
    return df

def n_linhas(tab, onde=""):
    return int(consultar_num(f"SELECT count() AS n FROM {tab}" + (f" WHERE {onde}" if onde else "")).n[0])

def existe(tab):
    return int(consultar_num(f"SELECT count() AS n FROM system.tables WHERE database = currentDatabase() AND name = {lit(tab)}").n[0]) > 0

def colunas_view(view):
    return consultar(f"SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(view)} ORDER BY position")["name"].tolist()

def ler_view(view, onde=""):
    # Lê uma view inteira. O id_linha (UInt64) fica como texto, para não perder precisão; o resto vira número.
    cols = colunas_view(view)
    sel = ", ".join(("toString(id_linha) AS id_linha" if c == "id_linha" else q(c)) for c in cols)
    df = consultar(f"SELECT {sel} FROM {view}" + (f" WHERE {onde}" if onde else ""))
    for c in df.columns:
        if c != "id_linha": df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def executar_robusto(sql, antes=None, tentativas=5, espera=120):
    # Escritas não são repetidas pelo transporte: se faltar memória no servidor (Code 241) ou a conexão cair, esperamos e refazemos (tudo aqui é idempotente).
    for k in range(1, tentativas + 1):
        try:
            if antes: executar(antes)
            return executar(sql)
        except PermissionError:
            raise
        except Exception as e:
            if isinstance(e, ErroConexao) or "241" in str(e) or "memory limit" in str(e).lower():
                print(f"  ⚠ falha por memória/conexão (tentativa {k}/{tentativas}): {str(e)[:160]}; aguardando {espera}s e refazendo...")
                time.sleep(espera); continue
            raise
    raise RuntimeError(f"não consegui concluir após {tentativas} tentativas: {sql[:100]}...")

def distribuir(total, tamanhos, alpha, tetos):
    # Reparte `total` linhas entre os tipos (alpha = 0: partes iguais), sem passar do teto de cada um; o que sobra de um tipo pequeno vai para os demais.
    q_ = {t: 0 for t in tamanhos}; livres = [t for t in tamanhos if tetos[t] > 0]; restante = int(total)
    if sum(tetos[t] for t in livres) < restante: raise ValueError(f"não há linhas suficientes: pedido {restante:,}, disponível {sum(tetos[t] for t in livres):,}")
    while restante > 0 and livres:
        w = {t: float(tamanhos[t]) ** alpha for t in livres}; sw = sum(w.values())
        alvo = {t: restante * w[t] / sw for t in livres}
        cheios = [t for t in livres if q_[t] + alvo[t] >= tetos[t]]
        if not cheios:
            base = {t: int(alvo[t]) for t in livres}; sobra = restante - sum(base.values())
            for t in sorted(livres, key=lambda t: alvo[t] - base[t], reverse=True)[:sobra]: base[t] += 1
            for t in livres: q_[t] += base[t]
            restante = 0
        else:
            for t in cheios:
                restante -= tetos[t] - q_[t]; q_[t] = tetos[t]; livres.remove(t)
    return q_

def _limpar(o):
    if isinstance(o, dict): return {str(k): _limpar(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_limpar(v) for v in o]
    if isinstance(o, np.ndarray): return _limpar(o.tolist())
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating, float)): return None if (o != o) else float(o)
    if isinstance(o, Path): return str(o)
    return o

def salvar_json(obj, caminho):
    Path(caminho).write_text(json.dumps(_limpar(obj), indent=1, ensure_ascii=False), encoding="utf-8")

In [5]:
# ---------- ferramentas de métricas, pré-processamento e custo ----------
def _div(a, b): return float(a) / float(b) if b else float("nan")

def metricas(y, p, limiar=0.5, w=None):
    # Todas as métricas de uma vez. Com `w` (pesos), as contagens e as métricas que dependem da proporção ficam ponderadas.
    # O ROC-AUC é calculado sem pesos: ele só compara ataques com BENIGN e não depende da proporção.
    y = np.asarray(y).astype(int); p = np.asarray(p, dtype=float); pred = (p >= limiar).astype(int)
    w_ = np.ones(len(y)) if w is None else np.asarray(w, dtype=float)
    tp = float(w_[(y == 1) & (pred == 1)].sum()); tn = float(w_[(y == 0) & (pred == 0)].sum())
    fp = float(w_[(y == 0) & (pred == 1)].sum()); fn = float(w_[(y == 1) & (pred == 0)].sum())
    rec, esp, prec = _div(tp, tp + fn), _div(tn, tn + fp), _div(tp, tp + fp)
    den = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    f1 = _div(2 * prec * rec, prec + rec) if (prec == prec and rec == rec) else float("nan")
    prec_b = _div(tn, tn + fn); f1_b = _div(2 * prec_b * esp, prec_b + esp) if (prec_b == prec_b and esp == esp) else float("nan")    # classe BENIGN como "positiva"
    return {"acuracia": _div(tp + tn, tp + tn + fp + fn), "acuracia_balanceada": (rec + esp) / 2, "precisao": prec, "recall": rec, "especificidade": esp,
            "taxa_falso_alarme": 1 - esp, "taxa_perda": 1 - rec, "f1": f1, "mcc": _div(tp * tn - fp * fn, den),
            "precisao_benign": prec_b, "f1_benign": f1_b, "f1_macro": (f1 + f1_b) / 2, "pr_auc_benign": float(average_precision_score(1 - y, 1 - p, sample_weight=w_)),
            "roc_auc": float(roc_auc_score(y, p)), "pr_auc": float(average_precision_score(y, p, sample_weight=w_)), "tp": tp, "fp": fp, "tn": tn, "fn": fn}

def recall_em_fpr(y, p, alvo):
    # Ponto de operação: o MAIOR recall com taxa de falso alarme <= alvo (limiar escolhido nestes dados).
    fpr, tpr, thr = roc_curve(np.asarray(y).astype(int), np.asarray(p, dtype=float))
    i = max(int(np.searchsorted(fpr, alvo, side="right")) - 1, 0)
    lim = float(thr[i]) if np.isfinite(thr[i]) else 1.0 + 1e-9
    return {"alvo_fpr": alvo, "limiar": lim, "recall": float(tpr[i]), "fpr": float(fpr[i])}

class Medidor:
    # Mede tempo de relógio, tempo de CPU (todos os núcleos do processo) e pico de RAM (RSS, amostrado a cada 50 ms) de um bloco `with`.
    def __init__(self, intervalo=0.05): self.intervalo = intervalo
    def __enter__(self):
        self.t0, self.c0 = time.perf_counter(), time.process_time(); self._ativo = psutil is not None
        if self._ativo:
            self._p = psutil.Process(); self.base = self._p.memory_info().rss; self.pico = self.base; self._parar = threading.Event()
            def amostrar():
                while not self._parar.is_set():
                    self.pico = max(self.pico, self._p.memory_info().rss); self._parar.wait(self.intervalo)
            self._th = threading.Thread(target=amostrar, daemon=True); self._th.start()
        return self
    def __exit__(self, *a):
        self.relogio, self.cpu = time.perf_counter() - self.t0, time.process_time() - self.c0
        if self._ativo:
            self._parar.set(); self._th.join(); self.pico = max(self.pico, self._p.memory_info().rss)
            self.pico_mib, self.delta_mib = self.pico / 2 ** 20, (self.pico - self.base) / 2 ** 20
        else:
            self.pico_mib = self.delta_mib = float("nan")

class PreprocessadorRF:
    # O pré-processamento do RF, igual ao da view definitiva: ±inf e vazios viram "ausente"; nas colunas com indicador, o ausente é trocado pela mediana do
    # TREINO e entra uma coluna <coluna>_ausente (0/1) no fim. A ordem das colunas do modelo é: atributos, depois os indicadores.
    def __init__(self, features, medianas, retiradas=(), versao_views=""):
        self.features, self.medianas, self.retiradas, self.versao_views = list(features), dict(medianas), list(retiradas), versao_views
        self.indicadores = [c for c in self.features if c in self.medianas]
        self.colunas_modelo = self.features + [c + "_ausente" for c in self.indicadores]
        idx = {c: i for i, c in enumerate(self.features)}
        self._pos = np.array([idx[c] for c in self.indicadores], dtype=int); self._med = np.array([self.medianas[c] for c in self.indicadores], dtype=float)
    def transformar(self, X):
        X = np.asarray(X, dtype=np.float64); X = np.where(np.isinf(X), np.nan, X)
        if len(self._pos) == 0: return X
        sub = X[:, self._pos]; aus = np.isnan(sub); X = X.copy(); X[:, self._pos] = np.where(aus, self._med, sub)
        return np.hstack([X, aus.astype(np.float64)])
    def para_dict(self):
        return {"features": self.features, "indicadores": self.indicadores, "colunas_modelo": self.colunas_modelo, "medianas_treino": self.medianas,
                "colunas_retiradas_ablacao": self.retiradas, "versao_views_rf": self.versao_views,
                "regra": "±inf e vazios = ausente; nas colunas com indicador: ausente -> mediana do treino e coluna <coluna>_ausente (0/1) no fim"}

def medir_inferencia(modelo, prep, X_modelo, X_raw, n_lat=1000, semente=42, repeticoes_lote=3):
    # Latência (1 fluxo por vez) e vazão (lote inteiro), com n_jobs = 1 e com o n_jobs original; com e sem o pré-processamento.
    rng = np.random.default_rng(semente); idx = rng.choice(len(X_modelo), size=min(n_lat, len(X_modelo)), replace=False)
    original = modelo.n_jobs; linhas = []
    for nj in dict.fromkeys([1, original]):
        modelo.set_params(n_jobs=nj)
        for i in idx[:20]: modelo.predict_proba(X_modelo[i:i + 1])                         # aquecimento
        t_mod, t_fim = [], []
        for i in idx:
            r = X_raw[i:i + 1]
            t0 = time.perf_counter(); m = prep.transformar(r); t1 = time.perf_counter(); modelo.predict_proba(m); t2 = time.perf_counter()
            t_mod.append(t2 - t1); t_fim.append(t2 - t0)
        l_mod, l_fim = np.percentile(t_mod, [50, 95, 99]) * 1000, np.percentile(t_fim, [50, 95, 99]) * 1000
        v_mod, v_fim = [], []
        for _ in range(repeticoes_lote):
            t0 = time.perf_counter(); modelo.predict_proba(X_modelo); v_mod.append(len(X_modelo) / (time.perf_counter() - t0))
            t0 = time.perf_counter(); modelo.predict_proba(prep.transformar(X_raw)); v_fim.append(len(X_raw) / (time.perf_counter() - t0))
        linhas.append({"n_jobs": nj, "n_latencia": len(idx), "latencia_modelo_p50_ms": l_mod[0], "latencia_modelo_p95_ms": l_mod[1], "latencia_modelo_p99_ms": l_mod[2],
                       "latencia_fim_a_fim_p50_ms": l_fim[0], "latencia_fim_a_fim_p95_ms": l_fim[1], "latencia_fim_a_fim_p99_ms": l_fim[2],
                       "vazao_modelo_fluxos_s": float(np.median(v_mod)), "vazao_fim_a_fim_fluxos_s": float(np.median(v_fim)), "n_lote": len(X_modelo)})
    modelo.set_params(n_jobs=original)
    return linhas

def tamanho_e_nos(modelo, caminho, compressao):
    joblib.dump(modelo, caminho, compress=compressao)
    return {"arquivo_mib": os.path.getsize(caminho) / 2 ** 20, "n_arvores": len(modelo.estimators_), "nos_total": int(sum(e.tree_.node_count for e in modelo.estimators_)),
            "profundidade_media": float(np.mean([e.get_depth() for e in modelo.estimators_])), "profundidade_max": int(max(e.get_depth() for e in modelo.estimators_))}

# autoteste das ferramentas (não toca no servidor)
_y = np.array([0, 0, 0, 0, 1, 1, 1, 1]); _p = np.array([.1, .2, .3, .6, .4, .7, .8, .9]); _m = metricas(_y, _p, 0.5)
assert (_m["tp"], _m["fp"], _m["tn"], _m["fn"]) == (3, 1, 3, 1) and abs(_m["mcc"] - 0.5) < 1e-12 and abs(_m["acuracia"] - 0.75) < 1e-12, _m
_o = recall_em_fpr(_y, _p, 0.25); assert abs(_o["recall"] - 1.0) < 1e-12 and _o["fpr"] <= 0.25 and abs(_o["limiar"] - 0.4) < 1e-12, _o
_m2 = metricas(_y, _p, 0.5); assert abs(_m2["precisao_benign"] - 0.75) < 1e-12 and abs(_m2["f1_benign"] - 0.75) < 1e-12 and abs(_m2["f1_macro"] - 0.75) < 1e-12, _m2
_o0 = recall_em_fpr(_y, _p, 0.0); assert abs(_o0["recall"] - 0.75) < 1e-12 and _o0["fpr"] == 0.0, _o0
_d = distribuir(100, {"a": 1000, "b": 1000, "c": 1000, "d": 10}, 0.0, {"a": 1000, "b": 1000, "c": 1000, "d": 10}); assert _d["d"] == 10 and sum(_d.values()) == 100, _d
print("Autoteste das ferramentas OK | sklearn", sklearn.__version__, "| psutil", "ausente" if psutil is None else psutil.__version__)

Autoteste das ferramentas OK | sklearn 1.9.1 | psutil 7.2.2


### Ferramentas do teste
- **`ler_csv_com_espera`**: lê o teste em CSV direto para a tabela (~200 MiB por 250 mil linhas, em vez de ~750 MiB pelo JSON), traz a coluna de **texto** com o tipo de ataque e, se o servidor ficar sem memória (`Code 241`), espera e repete.
- **`Barra`**: barra de progresso com tempo restante estimado.
- **`recall_por_tipo`**: recall de cada tipo de ataque (na primeira posição, a especificidade do BENIGN).
- **`prever_paralelo`**: os escores (distância à fronteira) em vários processos: o lote é dividido em pedaços e o resultado é **idêntico** ao sequencial.
- **`medir_inferencia_svm`**: latência de 1 fluxo (1 processo) e vazão em lote (1 e vários processos), com e sem o pré-processamento; o modelo já inclui a padronização.

In [6]:
import gc

def rss_mib():
    return psutil.Process().memory_info().rss / 2 ** 20 if psutil else float("nan")

def fmt_tempo(s):
    if s != s or s == float("inf"): return "?"
    s = int(round(s)); h, r = divmod(s, 3600); m, sg = divmod(r, 60)
    return f"{h}h {m:02d}min" if h else (f"{m}min {sg:02d}s" if m else f"{sg}s")

class Barra:
    # Barra de progresso: HTML no Jupyter/VS Code; se não houver IPython, imprime uma linha a cada 5% ou 60 s.
    def __init__(self, total, desc="", intervalo=0.4):
        self.total, self.desc, self.n, self.t0, self.intervalo = max(int(total), 1), desc, 0, time.perf_counter(), intervalo
        self._ult, self._ult_pct, self._extra, self._h = 0.0, -1, "", None
        try:
            from IPython.display import display as _d, HTML as _H
            self._H = _H; self._h = _d(_H(self._html()), display_id=True)
        except Exception:
            self._h = None
    def _info(self):
        dec = time.perf_counter() - self.t0; frac = min(self.n / self.total, 1.0)
        return frac, dec, (dec / self.n * (self.total - self.n) if self.n else float("nan"))
    def _linha(self):
        frac, dec, eta = self._info()
        return f"{self.desc}: {100 * frac:5.1f}% ({self.n}/{self.total}) | decorrido {fmt_tempo(dec)} | restante ~{fmt_tempo(eta)}" + (f" | {self._extra}" if self._extra else "")
    def _html(self):
        frac, dec, eta = self._info()
        return (f"<div style='font-family:monospace;font-size:13px'><b>{self.desc}</b> <progress value='{self.n}' max='{self.total}' style='width:260px;vertical-align:middle'></progress> "
                f"<b>{100 * frac:5.1f}%</b> ({self.n}/{self.total}) &nbsp;|&nbsp; decorrido {fmt_tempo(dec)} &nbsp;|&nbsp; restante ~{fmt_tempo(eta)}"
                + (f"<br><span style='color:#555'>{self._extra}</span>" if self._extra else "") + "</div>")
    def _mostrar(self, forcar=False):
        agora = time.perf_counter()
        if self._h is not None:
            if forcar or agora - self._ult >= self.intervalo: self._h.update(self._H(self._html())); self._ult = agora
        else:
            pct = int(100 * self.n / self.total) // 5
            if forcar or pct != self._ult_pct or agora - self._ult >= 60: print(self._linha(), flush=True); self._ult_pct, self._ult = pct, agora
    def avancar(self, n=1, extra=None):
        self.n += n
        if extra is not None: self._extra = extra
        self._mostrar(forcar=self.n >= self.total)
    def fechar(self, extra=""):
        self._extra = extra or self._extra; self._mostrar(forcar=True)
        if self._h is not None: print(f"{self.desc}: concluído em {fmt_tempo(time.perf_counter() - self.t0)}")


def recall_por_tipo(y, p, g, limiar, n_tipos):
    # Posição 0 = BENIGN (aqui vale a especificidade); posições seguintes = recall de cada tipo de ataque.
    y, g = np.asarray(y), np.asarray(g); pred = np.asarray(p) >= np.float32(limiar)
    pos = np.bincount(g[y == 1], minlength=n_tipos).astype(float); det = np.bincount(g[(y == 1) & pred], minlength=n_tipos).astype(float)
    r = np.divide(det, pos, out=np.full(n_tipos, np.nan), where=pos > 0); neg = float((y == 0).sum()); r[0] = float(((y == 0) & ~pred).sum()) / neg if neg else float("nan")
    return r

class ErroMemoriaServidor(Exception):
    pass


CH_SETTINGS_LEITURA = globals().get("CH_SETTINGS_LEITURA", {"max_threads": 1, "max_block_size": 8192})      # leitura leve para o servidor (menos colunas e blocos abertos ao mesmo tempo)
ESPERAS_MEMORIA = globals().get("ESPERAS_MEMORIA", (30, 60, 120, 240, 480))                            # segundos de espera após "sem memória" (Code 241) no servidor
LIMITE_MEM_SERVIDOR_GIB = globals().get("LIMITE_MEM_SERVIDOR_GIB", 2.0)                                        # volta a tentar antes do fim da espera se a memória do servidor cair abaixo disto


def ler_csv(query, colunas, texto=()):
    # Leitura em FLUXO: o servidor devolve CSV e o pandas lê direto da conexão, sem guardar o texto inteiro nem criar milhões de objetos Python
    # (o caminho JSON do consultar() gasta várias vezes mais memória). Mesmas regras de repetição do _http: só SELECT, com novas tentativas.
    global _conn
    if not re.match(r"\s*SELECT\b", query, re.I): raise PermissionError("ler_csv só executa SELECT")
    caminho = "/?" + urllib.parse.urlencode({"database": CH_DATABASE, "default_format": "CSV", "wait_end_of_query": 1, **CH_SETTINGS_LEITURA})
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    ultimo = None
    for espera in CONEXAO_ESPERAS:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s..."); time.sleep(espera)
        corpo = b""
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            if resp.status == 200:
                try:
                    return pd.read_csv(resp, header=None, names=colunas, dtype={c: (str if c in texto else np.float64) for c in colunas}, na_values=["\\N"])
                except pd.errors.EmptyDataError:
                    return pd.DataFrame({c: pd.Series(dtype=(object if c in texto else np.float64)) for c in colunas})
            corpo = resp.read()
        except (OSError, http.client.HTTPException, pd.errors.ParserError, ValueError) as e:
            ultimo = e; _fechar_conexao(); continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} corpo={corpo[:300].decode('utf-8', 'replace')!r}"); _fechar_conexao(); continue
        msg = corpo.decode("utf-8", "replace")
        if "MEMORY_LIMIT_EXCEEDED" in msg or msg.startswith("Code: 241"):
            raise ErroMemoriaServidor(msg[:600])
        raise Exception(msg[:2000])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(CONEXAO_ESPERAS)} tentativas: {ultimo!r}")


def _mem_servidor_gib():
    try:
        return float(consultar_num("SELECT toFloat64(value) AS v FROM system.asynchronous_metrics WHERE metric = 'MemoryResident'")["v"][0]) / 2 ** 30
    except Exception:
        return None


def ler_csv_com_espera(query, colunas, texto=()):
    # Se o servidor ficar sem memória (Code 241, comum num servidor de 3 GiB com merges em andamento), espera e tenta de novo; o erro é transitório.
    for k, espera in enumerate(list(ESPERAS_MEMORIA) + [None]):
        try:
            return ler_csv(query, colunas, texto)
        except ErroMemoriaServidor as e:
            if espera is None:
                raise ErroMemoriaServidor("O servidor continua sem memória depois de todas as esperas. Rode a célula de novo mais tarde: os lotes já prontos são reaproveitados. Último erro: " + str(e)[:300]) from None
            m = _mem_servidor_gib()
            print(f"  ⚠ o servidor ficou sem memória (Code 241; memória do servidor: {'?' if m is None else f'{m:.2f} GiB'}); esperando até {espera}s e tentando de novo ({k + 1}/{len(ESPERAS_MEMORIA)})...", flush=True)
            t0 = time.time()
            while time.time() - t0 < espera:
                time.sleep(min(15, espera))
                m = _mem_servidor_gib()
                if m is not None and m < LIMITE_MEM_SERVIDOR_GIB: break

_b = Barra(2, "autoteste"); _b.avancar(2); assert recall_por_tipo(np.array([0, 1, 1]), np.array([.1, .9, .2]), np.array([0, 1, 1]), 0.5, 2)[1] == 0.5
print("Autoteste das ferramentas do teste OK")

from joblib import Parallel, delayed

def _dec(modelo, X): return modelo.decision_function(X)
def prever_paralelo(modelo, X, k):
    # Escores (distância à fronteira) em k processos: o lote é dividido em k pedaços (o resultado é idêntico ao sequencial).
    if k <= 1: return modelo.decision_function(X)
    return np.concatenate(Parallel(n_jobs=k)(delayed(_dec)(modelo, c) for c in np.array_split(X, k)))

def medir_inferencia_svm(modelo, prep, X_modelo, X_raw, n_lat=1000, semente=42, n_lote=20_000, repeticoes_lote=2, k_par=2):
    # Latência (1 fluxo por vez, 1 processo) e vazão (lote) com 1 e com k_par processos; com e sem o pré-processamento (mediana do treino + indicadores). O modelo inclui a padronização.
    rng = np.random.default_rng(semente); idx = rng.choice(len(X_modelo), size=min(n_lat, len(X_modelo)), replace=False)
    for i in idx[:20]: modelo.decision_function(X_modelo[i:i + 1])                           # aquecimento
    t_mod, t_fim = [], []
    for i in idx:
        r = X_raw[i:i + 1]; t0 = time.perf_counter(); m = prep.transformar(r); t1 = time.perf_counter(); modelo.decision_function(m); t2 = time.perf_counter(); t_mod.append(t2 - t1); t_fim.append(t2 - t0)
    l_mod, l_fim = np.percentile(t_mod, [50, 95, 99]) * 1000, np.percentile(t_fim, [50, 95, 99]) * 1000
    Xl, Xr = X_modelo[:n_lote], X_raw[:n_lote]; linhas = []
    for k in dict.fromkeys([1, max(int(k_par), 1)]):
        if k > 1: Parallel(n_jobs=k)(delayed(time.sleep)(0.01) for _ in range(k))
        v_mod, v_fim = [], []
        for _ in range(repeticoes_lote):
            t0 = time.perf_counter(); prever_paralelo(modelo, Xl, k); v_mod.append(len(Xl) / (time.perf_counter() - t0))
            t0 = time.perf_counter(); prever_paralelo(modelo, prep.transformar(Xr), k); v_fim.append(len(Xr) / (time.perf_counter() - t0))
        nan = float("nan")
        linhas.append({"n_jobs": k, "n_latencia": len(idx), "latencia_modelo_p50_ms": l_mod[0] if k == 1 else nan, "latencia_modelo_p95_ms": l_mod[1] if k == 1 else nan, "latencia_modelo_p99_ms": l_mod[2] if k == 1 else nan,
                       "latencia_fim_a_fim_p50_ms": l_fim[0] if k == 1 else nan, "latencia_fim_a_fim_p95_ms": l_fim[1] if k == 1 else nan, "latencia_fim_a_fim_p99_ms": l_fim[2] if k == 1 else nan,
                       "vazao_modelo_fluxos_s": float(np.median(v_mod)), "vazao_fim_a_fim_fluxos_s": float(np.median(v_fim)), "n_lote": len(Xl)})
    return linhas
print("Ferramentas de previsão do SVM prontas")

Autoteste das ferramentas do teste OK
Ferramentas de previsão do SVM prontas


### Métricas leves em memória
Com ~7,2 milhões de previsões, as funções do sklearn ordenam cópias dos dados várias vezes (**~650 MiB** a mais para 3 limiares, no meu teste) e podem dar `MemoryError`. As métricas aqui saem de **contagens de ataques e BENIGN por valor de escore** (um RF de 100 árvores tem só ~100 valores): o resultado é **idêntico** ao do sklearn (diferença máxima de 2×10⁻¹⁶ nos meus testes) e gasta quase nenhuma memória. Na versão do SVM, o ponto de operação aceita escores fora de [0, 1].

In [7]:
_trap = getattr(np, "trapezoid", None) or getattr(np, "trapz")

def contagens_por_escore(y, p, bloco=1_000_000):
    # Conta ataques (y = 1) e BENIGN (y = 0) por VALOR DISTINTO do escore. É tudo o que as métricas precisam e ocupa memória mínima
    # (as funções do sklearn ordenam cópias de 7 milhões de elementos várias vezes). Os escores são comparados em float32, como foram gravados.
    y = np.asarray(y); p = np.asarray(p, dtype=np.float32)
    vals = np.unique(p)
    if len(vals) > 200_000:                                            # escores quase contínuos: caminho de uma passada
        vals, inv = np.unique(p, return_inverse=True)
        n_tot = np.bincount(inv, minlength=len(vals)); n_pos = np.bincount(inv, weights=y, minlength=len(vals))
    else:
        n_tot = np.zeros(len(vals), np.int64); n_pos = np.zeros(len(vals), np.float64)
        for i in range(0, len(p), bloco):
            idx = np.searchsorted(vals, p[i:i + bloco]); n_tot += np.bincount(idx, minlength=len(vals)); n_pos += np.bincount(idx, weights=y[i:i + bloco], minlength=len(vals))
    n_pos = np.rint(n_pos).astype(np.int64); n_tot = n_tot.astype(np.int64)
    return vals, n_pos, n_tot - n_pos

def _curva(tp_c, fp_c, P, N):
    tpr = np.concatenate(([0.0], tp_c / P)); fpr = np.concatenate(([0.0], fp_c / N)); rec = tp_c / P; prec = tp_c / (tp_c + fp_c)
    return fpr, tpr, float(_trap(tpr, fpr)), float(np.sum(np.diff(np.concatenate(([0.0], rec))) * prec))

def metricas_hist(vals, n_pos, n_neg, limiar=0.5):
    # Mesmas métricas de `metricas` (sem pesos), a partir das contagens por escore. Previsão = escore >= limiar (ambos em float32).
    m = vals >= np.float32(limiar)
    tp, fp, fn, tn = float(n_pos[m].sum()), float(n_neg[m].sum()), float(n_pos[~m].sum()), float(n_neg[~m].sum())
    P, N = tp + fn, tn + fp
    rec, esp, prec = _div(tp, P), _div(tn, N), _div(tp, tp + fp); prec_b = _div(tn, tn + fn)
    f1 = _div(2 * prec * rec, prec + rec) if (prec == prec and rec == rec) else float("nan")
    f1_b = _div(2 * prec_b * esp, prec_b + esp) if (prec_b == prec_b and esp == esp) else float("nan")
    den = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)); d = slice(None, None, -1)
    _, _, roc_auc, pr_a = _curva(np.cumsum(n_pos[d]), np.cumsum(n_neg[d]), P, N)       # ataque como positivo (escores em ordem decrescente)
    _, _, _, pr_b = _curva(np.cumsum(n_neg), np.cumsum(n_pos), N, P)                    # BENIGN como positivo (escore 1 - p: ordem crescente de p)
    return {"acuracia": _div(tp + tn, tp + tn + fp + fn), "acuracia_balanceada": (rec + esp) / 2, "precisao": prec, "recall": rec, "especificidade": esp,
            "taxa_falso_alarme": 1 - esp, "taxa_perda": 1 - rec, "f1": f1, "mcc": _div(tp * tn - fp * fn, den), "precisao_benign": prec_b, "f1_benign": f1_b,
            "f1_macro": (f1 + f1_b) / 2, "pr_auc_benign": pr_b, "roc_auc": roc_auc, "pr_auc": pr_a, "tp": tp, "fp": fp, "tn": tn, "fn": fn}

def recall_em_fpr_hist(vals, n_pos, n_neg, alvo):
    # O MAIOR recall com taxa de falso alarme <= alvo, lido na curva ROC exata (todos os pontos distintos).
    d = slice(None, None, -1); P, N = n_pos.sum(), n_neg.sum()
    tpr = np.concatenate(([0.0], np.cumsum(n_pos[d]) / P)); fpr = np.concatenate(([0.0], np.cumsum(n_neg[d]) / N)); thr = np.concatenate(([np.inf], vals[d].astype(float)))
    i = max(int(np.searchsorted(fpr, alvo, side="right")) - 1, 0); lim = float(thr[i]) if np.isfinite(thr[i]) else float(np.nextafter(vals.max(), np.inf))     # escores do SVM não ficam em [0, 1]
    return {"alvo_fpr": alvo, "limiar": lim, "recall": float(tpr[i]), "fpr": float(fpr[i])}

## 1. Carregar os modelos e conferir
Carrega o modelo final do SVM (notebook 11) e, se existir, o refit do padrão, e confere que usam as mesmas colunas, na mesma ordem, que a view de teste. Descobre também os **tipos de ataque do teste** e quais deles existem no treino.

In [8]:
# ---------- 1.1 Parâmetros e carga ----------
PASTA_BASE = Path(globals().get("REPO_ROOT", Path.cwd())) / "artefatos_modelos" / "svm_binario_busca"
PASTA_RUN = globals().get("PASTA_RUN", None)
if PASTA_RUN is None:
    runs = sorted(p for p in PASTA_BASE.glob("svm_busca_*") if (p / "modelo_final_refit.joblib").exists() and (p / "FINALIZADO").exists())
    assert runs, "nenhuma pasta svm_busca_* concluída com modelo_final_refit.joblib em " + str(PASTA_BASE) + ": rode antes o notebook 11"
    PASTA_RUN = runs[-1]
PASTA_RUN = Path(PASTA_RUN)
COMPARAR_COM_PADRAO = globals().get("COMPARAR_COM_PADRAO", True)
VIEW_TESTE, TAB_TESTE, TAB_TREINO = "prep_vw_rf_bin_teste_sem_sobrep", "prep_final_teste", "prep_final_treino"
LOTE_LINHAS = globals().get("LOTE_LINHAS", 50_000)          # ~40 MiB de memória por lote; a tabela é ordenada por id_linha, então muitos lotes pequenos são baratos para o servidor
PERMITIR_REPETIR = globals().get("PERMITIR_REPETIR", False)
PROCESSOS_PREVISAO = globals().get("PROCESSOS_PREVISAO", 2)
N_LATENCIA = globals().get("N_LATENCIA", 1000); N_LOTE_VAZAO = globals().get("N_LOTE_VAZAO", 20_000); REPETICOES_LOTE = globals().get("REPETICOES_LOTE", 2)
SEMENTE = globals().get("SEMENTE", 42)

RESULTADO = PASTA_RUN / "metricas_teste.json"
if RESULTADO.exists() and not PERMITIR_REPETIR:
    raise RuntimeError(f"O teste já foi avaliado para esta execução ({RESULTADO.name}). Repetir o teste depois de ajustar o modelo o invalida como avaliação independente. "
                       "Para repetir de propósito (por exemplo, para refazer os gráficos), defina PERMITIR_REPETIR = True antes desta célula.")
assert existe(VIEW_TESTE), f"a view {VIEW_TESTE} não existe: rode o notebook 03"

config = json.loads((PASTA_RUN / "config_treino.json").read_text(encoding="utf-8")); pc = json.loads((PASTA_RUN / "preprocessamento.json").read_text(encoding="utf-8")); hp = config["hiperparametros"]
lim_val = json.loads((PASTA_RUN / "limiares_validacao.json").read_text(encoding="utf-8")); LIMIAR = float(config["limiar_padrao"])
prep = PreprocessadorRF(pc["features"], pc["medianas_treino"], pc["colunas_retiradas_ablacao"], pc["versao_views_rf"]); CM = prep.colunas_modelo
modelo = joblib.load(PASTA_RUN / "modelo_final_refit.joblib"); svm_m = modelo.named_steps["svm"]
assert CM == pc["colunas_modelo"] == list(modelo.colunas_modelo_), "a ordem das colunas do modelo difere da gravada"
assert svm_m.kernel == "rbf" and svm_m.C == hp["C"] and np.isclose(svm_m._gamma, hp["gamma"]), "o modelo carregado não tem os hiperparâmetros gravados"
cols_view = colunas_view(VIEW_TESTE); assert cols_view[:2] == ["id_linha", "y"] and cols_view[2:] == CM, "as colunas da view de teste não coincidem com as do modelo (nome ou ordem)"
ESCOLHEU_PADRAO = hp["C"] == 1.0 and abs(hp["gamma_fator"] - 1.0) < 1e-12
COMPARAR = COMPARAR_COM_PADRAO and (PASTA_RUN / "modelo_padrao_refit.joblib").exists() and (PASTA_RUN / "limiares_validacao_padrao.json").exists()
if COMPARAR_COM_PADRAO and not COMPARAR: print("⚠ " + ("a escolhida já é o padrão: não há o que comparar." if ESCOLHEU_PADRAO else "o refit do padrão não foi gravado pelo notebook 11 (COMPARAR_COM_PADRAO estava desligado lá): a comparação será omitida."))
if COMPARAR:
    modelo_pad = joblib.load(PASTA_RUN / "modelo_padrao_refit.joblib"); lim_pad = json.loads((PASTA_RUN / "limiares_validacao_padrao.json").read_text(encoding="utf-8"))
    assert list(modelo_pad.colunas_modelo_) == CM, "o refit do padrão usa colunas diferentes"
    assert modelo_pad.named_steps["svm"].C == 1.0 and np.isclose(modelo_pad.named_steps["svm"]._gamma, hp["gamma_referencia_scale"]), "o refit do padrão não tem C = 1 e gamma = 1× o de referência"
N_TESTE = n_linhas(VIEW_TESTE)

# tipos de ataque do teste e se existem no treino (o prefixo DrDoS_ é retirado dos nomes)
nome_tipo = lambda t: t.replace("DrDoS_", "", 1) if t.startswith("DrDoS_") else t
tt = consultar_num(f"SELECT label_multi AS tipo, count() AS n FROM {TAB_TESTE} WHERE sobrepoe_treino = 0 GROUP BY label_multi"); tt["tipo"] = tt["tipo"].map(nome_tipo); tt = tt.groupby("tipo", as_index=False)["n"].sum()
tr = {nome_tipo(t) for t in consultar(f"SELECT DISTINCT label_multi AS tipo FROM {TAB_TREINO}")["tipo"]}
TIPOS = ["BENIGN"] + sorted(t for t in tt["tipo"] if t != "BENIGN"); COD = {t: i for i, t in enumerate(TIPOS)}; NT = len(TIPOS)
NO_TREINO = {t: (t in tr) for t in TIPOS}
print(f"Modelo avaliado: {PASTA_RUN.name} | treinado em {config['data']} | C = {hp['C']:g}, gamma = {hp['gamma']:.5f} ({hp['gamma_fator']:g}× o de referência) | {int(svm_m.n_support_.sum()):,} vetores de suporte")
if COMPARAR: print(f"Modelo padrão (comparação): C = 1, gamma = {modelo_pad.named_steps['svm']._gamma:.5f} (1×) | {int(modelo_pad.named_steps['svm'].n_support_.sum()):,} vetores de suporte")
print(f"Teste (sem sobreposição com o treino): {N_TESTE:,} linhas | tipos no teste: {NT - 1} ataques + BENIGN | previsão em {PROCESSOS_PREVISAO} processo(s)")
print("Tipos de ataque do teste que NÃO existem no treino:", [t for t in TIPOS if not NO_TREINO[t]] or "nenhum")
print(f"Limiar padrão do SVM: {LIMIAR:g} | limiares gravados na validação (escolhida):", [(p['alvo_fpr'], round(p['limiar'], 4)) for p in lim_val["pontos_de_operacao_validacao"]])
if COMPARAR: print("Limiares gravados na validação (padrão):", [(p['alvo_fpr'], round(p['limiar'], 4)) for p in lim_pad["pontos_de_operacao_validacao"]])

Modelo avaliado: svm_busca_20261025_004828 | treinado em 2026-10-25T01:48:34 | C = 100, gamma = 0.09521 (4× o de referência) | 2,661 vetores de suporte
Modelo padrão (comparação): C = 1, gamma = 0.02380 (1×) | 2,539 vetores de suporte
Teste (sem sobreposição com o treino): 7,219,189 linhas | tipos no teste: 7 ataques + BENIGN | previsão em 2 processo(s)
Tipos de ataque do teste que NÃO existem no treino: ['Portmap']
Limiar padrão do SVM: 0 | limiares gravados na validação (escolhida): [(0.01, -0.7444), (0.001, 0.9911)]
Limiares gravados na validação (padrão): [(0.01, -0.8443), (0.001, 1.0031)]


## 2. Previsão em lotes (com checkpoint)
Lê o teste em faixas de `id_linha`, junta o **tipo de ataque** de cada linha, prevê com o modelo ajustado (e com o padrão, para a comparação) e grava cada lote em `teste_lotes/`. Cada lote tem ~`LOTE_LINHAS` linhas (padrão **50.000**). A barra mostra o andamento, o tempo restante e a memória do processo; se o kernel morrer, o último texto da barra mostra em que lote foi. **Se a execução for interrompida, rode a célula de novo**: os lotes prontos são reaproveitados.

In [9]:
# ---------- 2.1 Previsão em lotes ----------
B = max(1, math.ceil(N_TESTE / LOTE_LINHAS)); PASTA_LOTES = PASTA_RUN / "teste_lotes"; PASTA_LOTES.mkdir(exist_ok=True)
cols_sql = ", ".join("v." + q(c) for c in CM); ys, gs, ps, pbs, t_pred, t_pred_pad, n_lote = [], [], [], [], [], [], []; X_AMOSTRA = None; reaproveitados = 0
barra = Barra(B, "Previsão no teste"); t_inicio = time.perf_counter()
for j in range(B):
    arq = PASTA_LOTES / f"lote_{j:03d}_de_{B:03d}.npz"
    if arq.exists():
        d = np.load(arq); ys.append(d["y"]); gs.append(d["g"]); ps.append(d["p"]); pbs.append(d["pb"]); t_pred.append(float(d["t"])); t_pred_pad.append(float(d["tb"])); n_lote.append(len(d["y"])); reaproveitados += 1
        barra.avancar(1, f"lote {j + 1}/{B}: já pronto"); continue
    lo, hi = j * 2 ** 64 // B, (j + 1) * 2 ** 64 // B
    onde_v = f"v.id_linha >= {lo}" + (f" AND v.id_linha < {hi}" if j < B - 1 else ""); onde_t = f"id_linha >= {lo}" + (f" AND id_linha < {hi}" if j < B - 1 else "")
    barra.avancar(0, f"lote {j + 1}/{B}: lendo... (memória do processo {rss_mib():,.0f} MiB)")
    df = ler_csv_com_espera(f"SELECT v.y, {cols_sql}, t.tipo FROM {VIEW_TESTE} AS v INNER JOIN (SELECT id_linha, label_multi AS tipo FROM {TAB_TESTE} WHERE {onde_t}) AS t ON v.id_linha = t.id_linha WHERE {onde_v}",
                            ["y"] + CM + ["tipo"], texto=("tipo",))
    if len(df) == 0:                                            # faixa sem linhas: registra o lote vazio e segue
        z8, z32 = np.zeros(0, np.int8), np.zeros(0, np.float32); np.savez(arq, y=z8, g=z8, p=z32, pb=z32, t=0.0, tb=0.0)
        ys.append(z8); gs.append(z8); ps.append(z32); pbs.append(z32); t_pred.append(0.0); t_pred_pad.append(0.0); n_lote.append(0); barra.avancar(1, f"lote {j + 1}/{B}: faixa sem linhas"); continue
    X, y = df[CM].to_numpy(float), df["y"].to_numpy(np.int8); assert np.isfinite(X).all(), "valores não finitos no teste"
    g = df["tipo"].map(nome_tipo).map(COD); assert g.notna().all(), "tipo de ataque do teste não reconhecido: " + str(sorted(set(df["tipo"].map(nome_tipo)) - set(COD))); g = g.to_numpy(np.int8)
    assert ((g == 0) == (y == 0)).all(), "o tipo BENIGN não coincide com y = 0 no teste"
    t0 = time.perf_counter(); p = prever_paralelo(modelo, X, PROCESSOS_PREVISAO).astype(np.float32); dt = time.perf_counter() - t0
    if COMPARAR: t0 = time.perf_counter(); pb = prever_paralelo(modelo_pad, X, PROCESSOS_PREVISAO).astype(np.float32); dtb = time.perf_counter() - t0
    else: pb, dtb = np.zeros(len(y), np.float32), 0.0
    np.savez(arq, y=y, g=g, p=p, pb=pb, t=dt, tb=dtb)
    ys.append(y); gs.append(g); ps.append(p); pbs.append(pb); t_pred.append(dt); t_pred_pad.append(dtb); n_lote.append(len(y))
    if X_AMOSTRA is None: X_AMOSTRA = X[:50_000].copy()
    barra.avancar(1, f"lote {j + 1}/{B}: {len(y):,} linhas ({dt:.1f} s de previsão, {len(y) / max(dt, 1e-9):,.0f} fluxos/s)")
    del df, X; gc.collect()
barra.fechar()
if reaproveitados: print(f"  {reaproveitados} lote(s) já estavam prontos e foram reaproveitados")
y_te, g_te, p_te, pb_te = np.concatenate(ys), np.concatenate(gs), np.concatenate(ps), np.concatenate(pbs)      # int8 e float32: 5 vezes menos memória que int64 e float64
del ys, gs, ps, pbs
assert len(y_te) == N_TESTE, f"as linhas dos lotes ({len(y_te):,}) não fecham com o total da view ({N_TESTE:,})"
print(f"\n✔ {len(y_te):,} previsões ({int((y_te == 0).sum()):,} BENIGN = {100 * (y_te == 0).mean():.3f}%) em {time.perf_counter() - t_inicio:.0f} s")

  ⚠ o servidor ficou sem memória (Code 241; memória do servidor: 1.63 GiB); esperando até 30s e tentando de novo (1/5)...
  ⚠ o servidor ficou sem memória (Code 241; memória do servidor: 1.70 GiB); esperando até 30s e tentando de novo (1/5)...
  ⚠ o servidor ficou sem memória (Code 241; memória do servidor: 1.67 GiB); esperando até 30s e tentando de novo (1/5)...
  ⚠ o servidor ficou sem memória (Code 241; memória do servidor: 1.67 GiB); esperando até 30s e tentando de novo (1/5)...
  ⚠ o servidor ficou sem memória (Code 241; memória do servidor: 1.67 GiB); esperando até 30s e tentando de novo (1/5)...
  ⚠ o servidor ficou sem memória (Code 241; memória do servidor: 1.77 GiB); esperando até 30s e tentando de novo (1/5)...
Previsão no teste: concluído em 44min 03s

✔ 7,219,189 previsões (56,216 BENIGN = 0.779%) em 2643 s


## 3. Métricas no teste
**Modelo ajustado** nos três limiares (o limiar 0 da fronteira do SVM e os dois da validação), com as matrizes de confusão e a linha de base do PR-AUC do BENIGN. Depois, a **comparação com o modelo padrão** nas mesmas linhas (cada modelo com os seus próprios limiares da validação). Os pontos lidos na curva do teste são **só informativos**: nenhum limiar é escolhido aqui. Por fim, o **recall por tipo de ataque** (na linha BENIGN, a especificidade).

In [10]:
# ---------- 3.1 Métricas ----------
PONTOS_VAL = lim_val["pontos_de_operacao_validacao"]
ordem = ["roc_auc", "recall", "taxa_falso_alarme", "especificidade", "acuracia_balanceada", "acuracia", "precisao", "f1", "mcc", "pr_auc", "precisao_benign", "f1_benign", "f1_macro", "pr_auc_benign"]
VALS, N_POS, N_NEG = contagens_por_escore(y_te, p_te)
print(f"Valores distintos de escore: {len(VALS)} | ataques: {int(N_POS.sum()):,} | BENIGN: {int(N_NEG.sum()):,}")
conjuntos = {f"limiar {LIMIAR}": metricas_hist(VALS, N_POS, N_NEG, LIMIAR)}
for pv in PONTOS_VAL: conjuntos[f"limiar da validação p/ falso alarme {pv['alvo_fpr']:.1%} ({pv['limiar']:.3f})"] = metricas_hist(VALS, N_POS, N_NEG, pv["limiar"])
tab_te = pd.DataFrame({nome: {k: m[k] for k in ordem} for nome, m in conjuntos.items()})
print("\n== Modelo ajustado: métricas no teste =="); display(tab_te.round(5))
cm_ = {nome: {k: int(m[k]) for k in ("tp", "fp", "tn", "fn")} for nome, m in conjuntos.items()}
print("\nMatrizes de confusão (VP = ataque detectado; FP = BENIGN chamado de ataque; VN = BENIGN correto; FN = ataque perdido):"); display(pd.DataFrame(cm_))
prev_benign = float(N_NEG.sum() / (N_POS.sum() + N_NEG.sum()))
print(f"\nLinha de base do PR-AUC do BENIGN = proporção de BENIGN no teste = {prev_benign:.5f} | o modelo ajustado obteve {conjuntos[f'limiar {LIMIAR}']['pr_auc_benign']:.5f}")
info = []
for alvo in (0.01, 0.001):                                      # APENAS informativo: limiar escolhido no próprio teste
    op = recall_em_fpr_hist(VALS, N_POS, N_NEG, alvo); info.append({"alvo_fpr": alvo, "limiar_no_teste": op["limiar"], "recall": op["recall"], "fpr": op["fpr"]})
print("\nPonto lido na curva ROC do TESTE (limiar escolhido no próprio teste; só informativo):"); display(pd.DataFrame(info).round(5))

# ---- comparação com o modelo padrão (descritiva) ----
ROT_LIM = [f"{LIMIAR:g}".replace(".", ",")] + [f"{p['alvo_fpr']:.1%} (validação)".replace(".", ",") for p in PONTOS_VAL]
if COMPARAR:
    VALS_B, NP_B, NN_B = contagens_por_escore(y_te, pb_te); PONTOS_PAD = lim_pad["pontos_de_operacao_validacao"]
    conjuntos_pad = {f"limiar {LIMIAR}": metricas_hist(VALS_B, NP_B, NN_B, LIMIAR)}
    for pv in PONTOS_PAD: conjuntos_pad[f"limiar da validação p/ falso alarme {pv['alvo_fpr']:.1%} ({pv['limiar']:.3f})"] = metricas_hist(VALS_B, NP_B, NN_B, pv["limiar"])
    METS = ["roc_auc", "recall", "taxa_falso_alarme", "mcc", "precisao_benign", "f1_benign", "f1_macro", "pr_auc_benign", "fp", "fn"]; comp = {}
    for rot_l, (nm, ma), (_, mb) in zip(ROT_LIM, conjuntos.items(), conjuntos_pad.items()):
        comp[f"ajustado @ {rot_l}"] = {k: ma[k] for k in METS}; comp[f"padrão @ {rot_l}"] = {k: mb[k] for k in METS}
    tab_comp_te = pd.DataFrame(comp); print("\n== Ajustado × padrão no teste (cada modelo com os limiares da própria validação; o limiar 0 é o mesmo) =="); display(tab_comp_te.round(5))
    dauc = conjuntos[f"limiar {LIMIAR}"]["roc_auc"] - conjuntos_pad[f"limiar {LIMIAR}"]["roc_auc"]
    print(f"ROC-AUC no teste: ajustado {conjuntos[f'limiar {LIMIAR}']['roc_auc']:.5f} × padrão {conjuntos_pad[f'limiar {LIMIAR}']['roc_auc']:.5f} (diferença {dauc:+.5f}). Comparação descritiva: a escolha foi feita na validação.")

# ---- recall por tipo de ataque ----
def rt(p, lim): return recall_por_tipo(y_te, p, g_te, lim, NT)
tab_tipo_te = pd.DataFrame({"n no teste": np.bincount(g_te, minlength=NT), "existe no treino?": ["sim" if NO_TREINO[t] else "NÃO" for t in TIPOS]}, index=TIPOS)
lims_aj = [LIMIAR] + [p["limiar"] for p in PONTOS_VAL]; lims_pad = [LIMIAR] + ([p["limiar"] for p in PONTOS_PAD] if COMPARAR else [])
for rot_l, lim in zip(ROT_LIM, lims_aj): tab_tipo_te[f"ajustado @ {rot_l}"] = rt(p_te, lim)
if COMPARAR:
    for rot_l, lim in zip(ROT_LIM, lims_pad): tab_tipo_te[f"padrão @ {rot_l}"] = rt(pb_te, lim)
tab_tipo_te["escore mediano (ajustado)"] = [float(np.median(p_te[g_te == i])) if (g_te == i).any() else float("nan") for i in range(NT)]
print("\n== Recall por tipo de ataque no teste (BENIGN = especificidade; limiares da validação de cada modelo) =="); display(tab_tipo_te.round(4))
pior = tab_tipo_te.iloc[1:][f"ajustado @ {ROT_LIM[0]}"].idxmin(); print(f"Tipo com menor recall no limiar {LIMIAR:g} (ajustado): {pior} ({tab_tipo_te.loc[pior, f'ajustado @ {ROT_LIM[0]}']:.4f}; {int(tab_tipo_te.loc[pior, 'n no teste']):,} linhas)")

Valores distintos de escore: 764976 | ataques: 7,162,973 | BENIGN: 56,216

== Modelo ajustado: métricas no teste ==


,limiar 0.0,limiar da validação p/ falso alarme 1.0% (-0.744),limiar da validação p/ falso alarme 0.1% (0.991)
roc_auc,0.99531,0.99531,0.99531
recall,0.99642,0.99970,0.59328
taxa_falso_alarme,0.01160,0.03239,0.00569
especificidade,0.98840,0.96761,0.99431
acuracia_balanceada,0.99241,0.98365,0.79379
acuracia,0.99636,0.99945,0.59640
precisao,0.99991,0.99975,0.99992
f1,0.99816,0.99972,0.74471
mcc,0.82075,0.96469,0.10496
pr_auc,0.99996,0.99996,0.99996



Matrizes de confusão (VP = ataque detectado; FP = BENIGN chamado de ataque; VN = BENIGN correto; FN = ataque perdido):


,limiar 0.0,limiar da validação p/ falso alarme 1.0% (-0.744),limiar da validação p/ falso alarme 0.1% (0.991)
tp,7137316,7160844,4249642
fp,652,1821,320
tn,55564,54395,55896
fn,25657,2129,2913331



Linha de base do PR-AUC do BENIGN = proporção de BENIGN no teste = 0.00779 | o modelo ajustado obteve 0.96823

Ponto lido na curva ROC do TESTE (limiar escolhido no próprio teste; só informativo):


,alvo_fpr,limiar_no_teste,recall,fpr
0,0.010,0.48189,0.73909,0.01000
1,0.001,1.27121,0.42572,0.00098



== Ajustado × padrão no teste (cada modelo com os limiares da própria validação; o limiar 0 é o mesmo) ==


,ajustado @ 0,padrão @ 0,"ajustado @ 1,0% (validação)","padrão @ 1,0% (validação)","ajustado @ 0,1% (validação)","padrão @ 0,1% (validação)"
roc_auc,0.99531,9.950400e-01,0.99531,0.99504,9.953100e-01,9.950400e-01
recall,0.99642,7.459500e-01,0.99970,0.99965,5.932800e-01,3.621000e-01
taxa_falso_alarme,0.01160,9.110000e-03,0.03239,0.02400,5.690000e-03,2.100000e-04
mcc,0.82075,1.477000e-01,0.96469,0.96575,1.049600e-01,6.630000e-02
precisao_benign,0.68411,2.970000e-02,0.96233,0.95614,1.883000e-02,1.215000e-02
f1_benign,0.80857,5.767000e-02,0.96496,0.96597,3.695000e-02,2.401000e-02
f1_macro,0.90337,4.560600e-01,0.98234,0.98285,3.908300e-01,2.778500e-01
pr_auc_benign,0.96823,9.730900e-01,0.96823,0.97309,9.682300e-01,9.730900e-01
fp,652.00000,5.120000e+02,1821.00000,1349.00000,3.200000e+02,1.200000e+01
fn,25657.00000,1.819788e+06,2129.00000,2517.00000,2.913331e+06,4.569245e+06


ROC-AUC no teste: ajustado 0.99531 × padrão 0.99504 (diferença +0.00026). Comparação descritiva: a escolha foi feita na validação.

== Recall por tipo de ataque no teste (BENIGN = especificidade; limiares da validação de cada modelo) ==


,n no teste,existe no treino?,ajustado @ 0,"ajustado @ 1,0% (validação)","ajustado @ 0,1% (validação)",padrão @ 0,"padrão @ 1,0% (validação)","padrão @ 0,1% (validação)",escore mediano (ajustado)
BENIGN,56216,sim,0.9884,0.9676,0.9943,0.9909,0.9760,0.9998,-1.3179
LDAP,112491,sim,0.9994,0.9995,0.9990,0.9992,0.9993,0.3233,0.9963
MSSQL,854242,sim,0.9997,0.9999,0.9994,0.9997,0.9998,0.9934,1.0320
NetBIOS,91175,sim,0.9994,0.9997,0.9969,0.9992,0.9996,0.9915,1.0393
Portmap,2702,NÃO,0.9567,0.9711,0.9378,0.9341,0.9634,0.7894,1.0393
Syn,4887060,sim,0.9949,0.9996,0.4041,0.6278,0.9996,0.0822,0.8097
UDP,1214746,sim,1.0000,1.0000,0.9999,0.9999,1.0000,0.9995,2.9931
UDPLag,557,sim,0.8815,0.8959,0.8618,0.8115,0.8833,0.6661,1.1767


Tipo com menor recall no limiar 0 (ajustado): UDPLag (0.8815; 557 linhas)


## 4. Custo computacional na inferência
**Vazão em dados reais** (todo o teste, com `PROCESSOS_PREVISAO` processos) e, numa amostra de 50 mil linhas do teste, a **latência de 1 fluxo** (1 processo) e a **vazão em lote** (1 e `PROCESSOS_PREVISAO` processos), com e sem o pré-processamento; o modelo já inclui a padronização. Para o padrão, a vazão em dados reais (mesmos lotes).

In [11]:
# ---------- 4.1 Custo de inferência ----------
t_total = float(sum(t_pred)); vazao_real = len(y_te) / t_total if t_total > 0 else float("nan")
print(f"Modelo ajustado, vazão em dados reais: {len(y_te):,} fluxos em {t_total:.1f} s de previsão = {vazao_real:,.0f} fluxos/s ({PROCESSOS_PREVISAO} processo(s); {int(modelo.named_steps['svm'].n_support_.sum()):,} vetores de suporte)")
t_total_pad = float(sum(t_pred_pad)); vazao_real_pad = len(y_te) / t_total_pad if (COMPARAR and t_total_pad > 0) else float("nan")
if COMPARAR: print(f"Modelo padrão,   vazão em dados reais: {len(y_te):,} fluxos em {t_total_pad:.1f} s de previsão = {vazao_real_pad:,.0f} fluxos/s ({PROCESSOS_PREVISAO} processo(s); {int(modelo_pad.named_steps['svm'].n_support_.sum()):,} vetores de suporte)")
if X_AMOSTRA is None:                                           # lotes já prontos de uma execução anterior: lê uma amostra só para a latência
    X_AMOSTRA = ler_csv_com_espera(f"SELECT {', '.join(q(c) for c in CM)} FROM {VIEW_TESTE} LIMIT 50000", CM).to_numpy(float)
X_raw = X_AMOSTRA[:, :len(prep.features)].copy()
for j, c in enumerate(prep.indicadores): X_raw[X_AMOSTRA[:, len(prep.features) + j] == 1, prep.features.index(c)] = np.nan
assert np.allclose(prep.transformar(X_raw), X_AMOSTRA), "o PreprocessadorRF não reproduz a matriz do teste"
inf_te = medir_inferencia_svm(modelo, prep, X_AMOSTRA, X_raw, n_lat=N_LATENCIA, semente=SEMENTE, n_lote=N_LOTE_VAZAO, repeticoes_lote=REPETICOES_LOTE, k_par=PROCESSOS_PREVISAO)
print("\nModelo ajustado: latência (1 fluxo) e vazão (lote) na amostra do teste"); display(pd.DataFrame(inf_te).set_index("n_jobs").T.round(3))

Modelo ajustado, vazão em dados reais: 7,219,189 fluxos em 1227.6 s de previsão = 5,881 fluxos/s (2 processo(s); 2,661 vetores de suporte)
Modelo padrão,   vazão em dados reais: 7,219,189 fluxos em 1174.5 s de previsão = 6,147 fluxos/s (2 processo(s); 2,539 vetores de suporte)

Modelo ajustado: latência (1 fluxo) e vazão (lote) na amostra do teste


n_jobs,1,2
n_latencia,1000.000,1000.000
latencia_modelo_p50_ms,1.371,NaN
latencia_modelo_p95_ms,1.670,NaN
latencia_modelo_p99_ms,2.305,NaN
latencia_fim_a_fim_p50_ms,1.410,NaN
latencia_fim_a_fim_p95_ms,1.723,NaN
latencia_fim_a_fim_p99_ms,2.376,NaN
vazao_modelo_fluxos_s,2559.846,5716.145
vazao_fim_a_fim_fluxos_s,2519.676,5701.045
n_lote,20000.000,20000.000


## 5. Gravação e conferências
Grava as previsões (com o tipo de cada fluxo), as métricas, o recall por tipo, a comparação com o padrão, o custo de inferência e os gráficos (curvas ROC e de precisão × recall do BENIGN; recall por tipo), e acrescenta uma linha ao `registro_testes.csv`. O `metricas_teste.json` é gravado por último: ele é o que aciona a trava de uso único.

In [12]:
# ---------- 5.1 Gravação ----------
np.savez_compressed(PASTA_RUN / "predicoes_teste.npz", y=y_te, g=g_te, p=p_te, pb=pb_te, tipos=np.array(TIPOS))
tab_te.reset_index(names="metrica").to_csv(PASTA_RUN / "metricas_teste.csv", index=False)
tab_tipo_te.reset_index(names="tipo").to_csv(PASTA_RUN / "recall_por_tipo_teste.csv", index=False)
if COMPARAR: tab_comp_te.reset_index(names="metrica").to_csv(PASTA_RUN / "comparacao_padrao_vs_ajustado_teste.csv", index=False)
salvar_json({"inferencia_teste": inf_te, "vazao_dados_reais_fluxos_s": vazao_real, "tempo_previsao_total_s": t_total, "processos_previsao": PROCESSOS_PREVISAO, "n_vetores_suporte": int(modelo.named_steps["svm"].n_support_.sum()),
             "padrao": {"vazao_dados_reais_fluxos_s": vazao_real_pad, "tempo_previsao_total_s": t_total_pad, "n_vetores_suporte": int(modelo_pad.named_steps["svm"].n_support_.sum())} if COMPARAR else None}, PASTA_RUN / "custo_inferencia_teste.json")
def curvas(NP, NN):
    d_ = slice(None, None, -1); fpr_ = np.concatenate(([0.0], np.cumsum(NN[d_]) / NN.sum())); tpr_ = np.concatenate(([0.0], np.cumsum(NP[d_]) / NP.sum()))
    tp_b = np.cumsum(NN); return fpr_, tpr_, tp_b / NN.sum(), tp_b / (tp_b + np.cumsum(NP))                 # (falso alarme, recall dos ataques, recall do BENIGN, precisão do BENIGN)
try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.2)); fpr_, tpr_, rc_, pr_ = curvas(N_POS, N_NEG)
    ax[0].plot(fpr_, tpr_, label="ajustado"); ax[1].plot(rc_, pr_, label="ajustado")
    if COMPARAR: f2, t2, r2, p2 = curvas(NP_B, NN_B); ax[0].plot(f2, t2, "--", label="padrão (C = 1, gamma = 1×)"); ax[1].plot(r2, p2, "--", label="padrão (C = 1, gamma = 1×)")
    ax[0].set_xscale("symlog", linthresh=1e-4); ax[0].set_xlabel("taxa de falso alarme (BENIGN chamado de ataque)"); ax[0].set_ylabel("recall dos ataques"); ax[0].set_title("Curva ROC (teste)"); ax[0].grid(alpha=.3); ax[0].legend()
    ax[1].axhline(prev_benign, ls=":", c="gray", label=f"linha de base ({prev_benign:.4f})"); ax[1].set_xlabel("recall do BENIGN"); ax[1].set_ylabel("precisão do BENIGN"); ax[1].set_title("Precisão × recall do BENIGN (teste)"); ax[1].legend(); ax[1].grid(alpha=.3)
    fig.tight_layout(); fig.savefig(PASTA_RUN / "curvas_teste.png", dpi=130); plt.close(fig)
    cols_g = [c for c in tab_tipo_te.columns if c.endswith(f"@ {ROT_LIM[0]}") or c.endswith(f"@ {ROT_LIM[-1]}")]; ataques = tab_tipo_te.iloc[1:]
    fig, ax = plt.subplots(figsize=(max(9, 0.9 * len(ataques) + 3.5), 4.4)); larg = 0.8 / max(len(cols_g), 1)
    for i, c in enumerate(cols_g): ax.bar(np.arange(len(ataques)) + i * larg, ataques[c], larg, label=c)
    ax.set_xticks(np.arange(len(ataques)) + 0.4 - larg / 2); ax.set_xticklabels([f"{t}{'' if NO_TREINO[t] else ' *'}" for t in ataques.index], rotation=45, ha="right"); ax.set_ylabel("recall"); ax.set_ylim(0, 1.05)
    ax.set_title("Recall por tipo de ataque no teste (* = tipo que não existe no treino)"); ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1.01, 1.0)); ax.grid(axis="y", alpha=.3); fig.savefig(PASTA_RUN / "recall_por_tipo_teste.png", dpi=130, bbox_inches="tight"); plt.close(fig)
    print("✔ gráficos salvos: curvas_teste.png, recall_por_tipo_teste.png")
except ImportError:
    print("(matplotlib não está instalado: gráficos não gerados)")
m0 = conjuntos[f"limiar {LIMIAR}"]
registro = {"run_id": config["run_id"], "data_teste": datetime.now().isoformat(timespec="seconds"), "n_teste": len(y_te), "benign_teste": int((y_te == 0).sum()), "limiar": LIMIAR,
            "modelo": "SVM (RBF)", "C": hp["C"], "gamma": hp["gamma"], "n_vetores_suporte": int(modelo.named_steps["svm"].n_support_.sum()), "processos_previsao": PROCESSOS_PREVISAO,
            **{k: m0[k] for k in ("roc_auc", "recall", "taxa_falso_alarme", "especificidade", "acuracia_balanceada", "acuracia", "precisao", "f1", "mcc", "pr_auc", "precisao_benign", "f1_benign", "f1_macro", "pr_auc_benign")},
            "roc_auc_padrao": conjuntos_pad[f"limiar {LIMIAR}"]["roc_auc"] if COMPARAR else float("nan"), "vazao_dados_reais_fluxos_s": vazao_real, "pasta": str(PASTA_RUN)}
reg_csv = PASTA_BASE / "registro_testes.csv"; novo_reg = pd.DataFrame([registro])
if reg_csv.exists(): novo_reg = pd.concat([pd.read_csv(reg_csv)[lambda d: d["run_id"].astype(str) != str(registro["run_id"])], novo_reg], ignore_index=True)        # repetir o teste (PERMITIR_REPETIR) substitui a linha da mesma execução
novo_reg.to_csv(reg_csv, index=False)
salvar_json({"run_id": config["run_id"], "n_teste": len(y_te), "limiar_padrao": LIMIAR, "hiperparametros": config["hiperparametros"], "metricas": conjuntos, "linha_de_base_pr_auc_benign": prev_benign,
             "pontos_informativos_no_teste": info, "limiares_da_validacao": PONTOS_VAL, "metricas_padrao": conjuntos_pad if COMPARAR else None, "limiares_padrao": PONTOS_PAD if COMPARAR else None,
             "recall_por_tipo": tab_tipo_te.round(6).to_dict(orient="index"), "observacao": "teste sem sobreposição com o treino, sem balancear e sem pesos; comparação com o padrão é descritiva"}, RESULTADO)

ok = []
def verifica(nome, cond, detalhe=""):
    ok.append(bool(cond)); print(("✔" if cond else "✘"), nome, detalhe)
cm_att = tab_tipo_te.iloc[1:]
verifica("Previsões = linhas do teste", len(y_te) == N_TESTE, f"({len(y_te):,})")
verifica("Escores finitos em todos os modelos", bool(np.isfinite(p_te).all()) and (not COMPARAR or bool(np.isfinite(pb_te).all())))
verifica("Matrizes de confusão fecham com o total", all(sum(v.values()) == N_TESTE for v in cm_.values()))
verifica("Lotes gravados somam o teste", sum(n_lote) == N_TESTE and len(list(PASTA_LOTES.glob(f"lote_*_de_{B:03d}.npz"))) == B, f"({B} lotes)")
verifica("Modelo final avaliado é o gravado pelo notebook 11", config["run_id"] == PASTA_RUN.name.replace("svm_busca_", ""))
verifica("Todo tipo do teste foi reconhecido e a soma por tipo fecha com o total", int(tab_tipo_te["n no teste"].sum()) == N_TESTE and bool((tab_tipo_te["n no teste"] > 0).all()))
verifica(f"Recall por tipo é consistente com o recall global (limiar {LIMIAR:g})", abs(float((cm_att[f"ajustado @ {ROT_LIM[0]}"] * cm_att["n no teste"]).sum()) - m0["tp"]) < 0.5)
verifica("Arquivos gravados", all((PASTA_RUN / f).exists() for f in ("predicoes_teste.npz", "metricas_teste.csv", "recall_por_tipo_teste.csv", "custo_inferencia_teste.json", "metricas_teste.json")))
_rel = np.load(PASTA_RUN / "predicoes_teste.npz"); _v, _a, _b = contagens_por_escore(_rel["y"], _rel["p"])
verifica("Relido: predicoes_teste.npz reproduz as métricas", abs(metricas_hist(_v, _a, _b, LIMIAR)["mcc"] - m0["mcc"]) < 1e-9)
assert all(ok), "há checagens reprovadas acima"
print(f"\n✔ {sum(ok)}/{len(ok)} checagens aprovadas. Resultados em {PASTA_RUN}")

✔ gráficos salvos: curvas_teste.png, recall_por_tipo_teste.png
✔ Previsões = linhas do teste (7,219,189)
✔ Escores finitos em todos os modelos 
✔ Matrizes de confusão fecham com o total 
✔ Lotes gravados somam o teste (145 lotes)
✔ Modelo final avaliado é o gravado pelo notebook 11 
✔ Todo tipo do teste foi reconhecido e a soma por tipo fecha com o total 
✔ Recall por tipo é consistente com o recall global (limiar 0) 
✔ Arquivos gravados 
✔ Relido: predicoes_teste.npz reproduz as métricas 

✔ 9/9 checagens aprovadas. Resultados em C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\svm_binario_busca\svm_busca_20261025_004828
